# Baseline results: Hallucination, Overall Metrics, Numeric Tolerance

Compares **Gemini 2.5 Flash (native)**, **Gemini 2.5 Flash (free-form)**, and **LandingAI (ADE)**.
Three tables are produced:
- Hallucination rate (empty GT but non-empty prediction)
- Overall correctness/completeness/overall score
- Numeric tolerance score


In [1]:
import json
from pathlib import Path
import pandas as pd

# Resolve experiment-scripts directory (run from repo root or experiment-analysis/)
_candidates = [
    Path.cwd() / 'experiment-scripts',
    (Path.cwd() / '..' / 'experiment-scripts').resolve(),
    Path.cwd(),
]
SCRIPT_BASE = next((p for p in _candidates if p.exists()), None)
if SCRIPT_BASE is None:
    raise FileNotFoundError('Could not find experiment-scripts. Run from repo root or experiment-analysis/.')

# Three baselines: Gemini 2.5 native, Gemini 2.5 free-form, Landing AI
BASELINE_RUNS = [
    ('Gemini 2.5 Flash (native)', SCRIPT_BASE / 'baselines_file_search_results' / 'gemini_native' / 'gemini-2.5-flash'),
    ('Gemini 2.5 Flash (free-form)', SCRIPT_BASE / 'baselines_file_search_results' / 'free_form' / 'gemini-2.5-flash'),
    ('LandingAI (ADE)', SCRIPT_BASE / 'baselines_landing_ai_new_results'),
]

def load_eval_results(trial_dir: Path):
    path = trial_dir / 'evaluation' / 'evaluation_results.json'
    if not path.exists():
        return None
    return json.loads(path.read_text(encoding='utf-8'))

def is_empty(val) -> bool:
    if val is None:
        return True
    s = str(val).strip().lower().strip(" .,:;")
    return s in {
        "", "nan", "not applicable", "not reported", "not present",
        "na", "n/a", "not found", "not available", "unknown"
    }


# Use PDFs common to all runs
common_pdfs = None
for _, run_path in BASELINE_RUNS:
    if not run_path.exists():
        continue
    pdfs = {d.name for d in run_path.iterdir() if d.is_dir()}
    common_pdfs = pdfs if common_pdfs is None else common_pdfs | pdfs
common_pdfs = sorted(common_pdfs or [])

rows = []
for model_label, run_path in BASELINE_RUNS:
    if not run_path.exists():
        continue
    for pdf in common_pdfs:
        data = load_eval_results(run_path / pdf)
        if data is None:
            continue
        cols = data.get('columns', {})
        empty_gt = 0
        halluc = 0
        for _, col_data in cols.items():
            gt = col_data.get('ground_truth', '')
            pred = col_data.get('predicted', '')
            if is_empty(gt):
                empty_gt += 1
                if not is_empty(pred):
                    halluc += 1
        rate = (halluc / empty_gt) if empty_gt else None
        rows.append({
            'model': model_label,
            'pdf': pdf,
            'hallucination_rate': rate,
        })

df = pd.DataFrame(rows)
pivot = df.pivot(index='pdf', columns='model', values='hallucination_rate')
pivot.loc['AVG'] = pivot.mean(numeric_only=True)
display((pivot * 100).round(1))


model,Gemini 2.5 Flash (free-form),Gemini 2.5 Flash (native),LandingAI (ADE)
pdf,,,
NCT00104715_Gravis_GETUG_EU'15,21.2,21.2,0.0
NCT00268476_Attard_STAMPEDE_Lancet'23,20.5,13.5,11.5
NCT00268476_James_STAMPEDE_IJC'22,2.9,0.0,5.9
NCT00309985_Kriayako_CHAARTED_JCO'18,12.9,12.9,9.4
NCT00309985_Sweeney_CHAARTED_NEJM'15,20.8,11.3,6.0
NCT01809691_Aggarwal_SWOG1216_JCO'22,9.8,4.5,0.0
NCT01957436_Fizazi_PEACE1_Lancet'22,25.0,14.3,1.5
NCT02446405_Sweeney_ENZAMET_Lancet Onc'23,21.4,23.5,20.6
NCT02799602_Hussain_ARASENS_JCO'23,19.5,11.4,26.3


In [2]:
import json
from pathlib import Path
import pandas as pd

_candidates = [
    Path.cwd() / 'experiment-scripts',
    (Path.cwd() / '..' / 'experiment-scripts').resolve(),
    Path.cwd(),
]
SCRIPT_BASE = next((p for p in _candidates if p.exists()), None)
if SCRIPT_BASE is None:
    raise FileNotFoundError('Could not find experiment-scripts. Run from repo root or experiment-analysis/.')

# Three baselines: Gemini 2.5 native, Gemini 2.5 free-form, Landing AI
BASELINE_RUNS = [
    ('Gemini 2.5 Flash (native)', SCRIPT_BASE / 'baselines_file_search_results' / 'gemini_native' / 'gemini-2.5-flash'),
    ('Gemini 2.5 Flash (free-form)', SCRIPT_BASE / 'baselines_file_search_results' / 'free_form' / 'gemini-2.5-flash'),
    ('LandingAI (ADE)', SCRIPT_BASE / 'baselines_landing_ai_new_results'),
]

def load_eval_results(trial_dir: Path):
    path = trial_dir / 'evaluation' / 'evaluation_results.json'
    if not path.exists():
        return None
    return json.loads(path.read_text(encoding='utf-8'))

def compute_overall_from_eval(data: dict):
    cols = data.get('columns', {})
    if not cols:
        return None
    correctness = [v.get('correctness', 0) for v in cols.values()]
    completeness = [v.get('completeness', 0) for v in cols.values()]
    overall = [v.get('overall', 0) for v in cols.values()]
    return {
        'avg_correctness': sum(correctness) / len(correctness),
        'avg_completeness': sum(completeness) / len(completeness),
        'avg_overall': sum(overall) / len(overall),
    }

common_pdfs = None
for _, run_path in BASELINE_RUNS:
    if not run_path.exists():
        continue
    pdfs = {d.name for d in run_path.iterdir() if d.is_dir()}
    common_pdfs = pdfs if common_pdfs is None else common_pdfs | pdfs
common_pdfs = sorted(common_pdfs or [])

rows = []
for model_label, run_path in BASELINE_RUNS:
    if not run_path.exists():
        continue
    for pdf in common_pdfs:
        data = load_eval_results(run_path / pdf)
        if data is None:
            continue
        overall = compute_overall_from_eval(data)
        if overall is None:
            continue
        rows.append({
            'model': model_label,
            'pdf': pdf,
            **overall,
        })

df = pd.DataFrame(rows)
pivot = df.pivot(index='pdf', columns='model', values=['avg_correctness','avg_completeness','avg_overall'])
display((pivot * 100).round(1))


avg_correctness  \
model                                     Gemini 2.5 Flash (free-form)   
pdf                                                                      
NCT00104715_Gravis_GETUG_EU'15                                    84.4   
NCT00268476_Attard_STAMPEDE_Lancet'23                             72.3   
NCT00268476_James_STAMPEDE_IJC'22                                 89.4   
NCT00309985_Kriayako_CHAARTED_JCO'18                              70.1   
NCT00309985_Sweeney_CHAARTED_NEJM'15                              80.1   
NCT01809691_Aggarwal_SWOG1216_JCO'22                              79.4   
NCT01957436_Fizazi_PEACE1_Lancet'22                               69.3   
NCT02446405_Sweeney_ENZAMET_Lancet Onc'23                         72.6   
NCT02799602_Hussain_ARASENS_JCO'23                                83.5   
NCT02799602_Smith_ARASENS_NEJM'22                                 84.6   

                                                                     \
model                                     Gemini 2.5 Flash (native)   
pdf                                                                   
NCT00104715_Gravis_GETUG_EU'15                                 80.5   
NCT00268476_Attard_STAMPEDE_Lancet'23                          80.9   
NCT00268476_James_STAMPEDE_IJC'22                              92.8   
NCT00309985_Kriayako_CHAARTED_JCO'18                           72.3   
NCT00309985_Sweeney_CHAARTED_NEJM'15                           88.6   
NCT01809691_Aggarwal_SWOG1216_JCO'22                           85.9   
NCT01957436_Fizazi_PEACE1_Lancet'22                            73.7   
NCT02446405_Sweeney_ENZAMET_Lancet Onc'23                      75.0   
NCT02799602_Hussain_ARASENS_JCO'23                             84.9   
NCT02799602_Smith_ARASENS_NEJM'22                              89.4   

                                                           \
model                                     LandingAI (ADE)   
pdf                                                         
NCT00104715_Gravis_GETUG_EU'15                       92.5   
NCT00268476_Attard_STAMPEDE_Lancet'23                79.3   
NCT00268476_James_STAMPEDE_IJC'22                    87.1   
NCT00309985_Kriayako_CHAARTED_JCO'18                 67.2   
NCT00309985_Sweeney_CHAARTED_NEJM'15                 91.6   
NCT01809691_Aggarwal_SWOG1216_JCO'22                 85.0   
NCT01957436_Fizazi_PEACE1_Lancet'22                  76.4   
NCT02446405_Sweeney_ENZAMET_Lancet Onc'23            86.8   
NCT02799602_Hussain_ARASENS_JCO'23                   64.0   
NCT02799602_Smith_ARASENS_NEJM'22                    83.3   

                                                      avg_completeness  \
model                                     Gemini 2.5 Flash (free-form)   
pdf                                                                      
NCT00104715_Gravis_GETUG_EU'15                                    82.8   
NCT00268476_Attard_STAMPEDE_Lancet'23                             71.5   
NCT00268476_James_STAMPEDE_IJC'22                                 91.7   
NCT00309985_Kriayako_CHAARTED_JCO'18                              66.8   
NCT00309985_Sweeney_CHAARTED_NEJM'15                              80.1   
NCT01809691_Aggarwal_SWOG1216_JCO'22                              79.4   
NCT01957436_Fizazi_PEACE1_Lancet'22                               71.6   
NCT02446405_Sweeney_ENZAMET_Lancet Onc'23                         71.8   
NCT02799602_Hussain_ARASENS_JCO'23                                83.5   
NCT02799602_Smith_ARASENS_NEJM'22                                 84.6   

                                                                     \
model                                     Gemini 2.5 Flash (native)   
pdf                                                                   
NCT00104715_Gravis_GETUG_EU'15                                 80.1   
NCT00268476_Attard_STAMPEDE_Lancet'23                          77.3   
NCT00268476_James_STAMPEDE_IJC'22                       

In [3]:
import json
from pathlib import Path
import pandas as pd

_candidates = [
    Path.cwd() / 'experiment-scripts',
    (Path.cwd() / '..' / 'experiment-scripts').resolve(),
    Path.cwd(),
]
SCRIPT_BASE = next((p for p in _candidates if p.exists()), None)
if SCRIPT_BASE is None:
    raise FileNotFoundError('Could not find experiment-scripts. Run from repo root or experiment-analysis/.')

# Three baselines: Gemini 2.5 native, Gemini 2.5 free-form, Landing AI
BASELINE_RUNS = [
    ('Gemini 2.5 Flash (native)', SCRIPT_BASE / 'baselines_file_search_results' / 'gemini_native' / 'gemini-2.5-flash'),
    ('Gemini 2.5 Flash (free-form)', SCRIPT_BASE / 'baselines_file_search_results' / 'free_form' / 'gemini-2.5-flash'),
    ('LandingAI (ADE)', SCRIPT_BASE / 'baselines_landing_ai_new_results'),
]

def load_eval_results(trial_dir: Path):
    path = trial_dir / 'evaluation' / 'evaluation_results.json'
    if not path.exists():
        return None
    return json.loads(path.read_text(encoding='utf-8'))

def numeric_tolerance_score(data: dict):
    cols = data.get('columns', {})
    vals = [v.get('overall', 0) for v in cols.values() if v.get('category') == 'numeric_tolerance']
    if not vals:
        return None
    return sum(vals) / len(vals)

common_pdfs = None
for _, run_path in BASELINE_RUNS:
    if not run_path.exists():
        continue
    pdfs = {d.name for d in run_path.iterdir() if d.is_dir()}
    common_pdfs = pdfs if common_pdfs is None else common_pdfs | pdfs
common_pdfs = sorted(common_pdfs or [])

rows = []
for model_label, run_path in BASELINE_RUNS:
    if not run_path.exists():
        continue
    for pdf in common_pdfs:
        data = load_eval_results(run_path / pdf)
        if data is None:
            continue
        score = numeric_tolerance_score(data)
        rows.append({
            'model': model_label,
            'pdf': pdf,
            'numeric_tolerance_avg_overall': score,
        })

df = pd.DataFrame(rows)
pivot = df.pivot(index='pdf', columns='model', values='numeric_tolerance_avg_overall')
pivot.loc['AVG'] = pivot.mean(numeric_only=True)
display((pivot * 100).round(1))


model,Gemini 2.5 Flash (free-form),Gemini 2.5 Flash (native),LandingAI (ADE)
pdf,,,
NCT00104715_Gravis_GETUG_EU'15,83.3,78.5,96.3
NCT00268476_Attard_STAMPEDE_Lancet'23,71.4,79.9,82.2
NCT00268476_James_STAMPEDE_IJC'22,91.5,96.2,90.6
NCT00309985_Kriayako_CHAARTED_JCO'18,64.5,68.6,68.1
NCT00309985_Sweeney_CHAARTED_NEJM'15,83.6,89.3,94.9
NCT01809691_Aggarwal_SWOG1216_JCO'22,79.0,87.6,88.8
NCT01957436_Fizazi_PEACE1_Lancet'22,68.9,71.5,80.9
NCT02446405_Sweeney_ENZAMET_Lancet Onc'23,71.0,73.6,86.9
NCT02799602_Hussain_ARASENS_JCO'23,82.2,85.2,68.4


In [4]:
import json
from pathlib import Path
import pandas as pd

# Resolve experiment-scripts directory
_candidates = [
    Path.cwd() / "experiment-scripts",
    (Path.cwd() / ".." / "experiment-scripts").resolve(),
    Path.cwd(),
]
SCRIPT_BASE = next((p for p in _candidates if p.exists()), None)
if SCRIPT_BASE is None:
    raise FileNotFoundError("Could not find experiment-scripts. Run from repo root or experiment-analysis/.")

# Three baselines: Gemini 2.5 native, Gemini 2.5 free-form, Landing AI
BASELINE_RUNS = [
    ("Gemini 2.5 Flash (native)", SCRIPT_BASE / "baselines_file_search_results" / "gemini_native" / "gemini-2.5-flash"),
    ("Gemini 2.5 Flash (free-form)", SCRIPT_BASE / "baselines_file_search_results" / "free_form" / "gemini-2.5-flash"),
    ("LandingAI (ADE)", SCRIPT_BASE / "baselines_landing_ai_new_results"),
]

def load_eval_results(trial_dir: Path):
    p = trial_dir / "evaluation" / "evaluation_results.json"
    if not p.exists():
        return None
    return json.loads(p.read_text(encoding="utf-8"))

# Use PDFs common to all runs
common_pdfs = None
for _, run_path in BASELINE_RUNS:
    if not run_path.exists():
        continue
    pdfs = {d.name for d in run_path.iterdir() if d.is_dir()}
    common_pdfs = pdfs if common_pdfs is None else common_pdfs & pdfs
common_pdfs = sorted(common_pdfs or [])

rows = []
for model_label, run_path in BASELINE_RUNS:
    if not run_path.exists():
        continue
    for pdf in common_pdfs:
        data = load_eval_results(run_path / pdf)
        if data is None:
            continue
        cols = data.get("columns", {})
        for col_name, col_data in cols.items():
            rows.append({
                "pdf": pdf,
                "column": col_name,
                "model": model_label,
                "category": col_data.get("category"),
                "ground_truth": col_data.get("ground_truth"),
                "predicted": col_data.get("predicted"),
                "correctness": col_data.get("correctness"),
                "completeness": col_data.get("completeness"),
                "overall": col_data.get("overall"),
                "reason": col_data.get("reason"),
            })

df = pd.DataFrame(rows)

# FAILURE CRITERIA (edit as needed)
# failure = overall < 1.0
failures = df[df["overall"] < 1.0].copy()

# Optional: sort to inspect worst first
failures = failures.sort_values(["overall", "pdf", "column", "model"])

display(failures.reset_index(drop=True))
failures.to_csv("failures.csv", index=False)


,pdf,column,model,category,ground_truth,predicted,correctness,completeness,overall,reason
0,NCT00104715_Gravis_GETUG_EU'15,Add-on Treatment,LandingAI (ADE),structured_text,Docetaxel,Not reported,0.0,0.0,0.00,The prediction fails to identify the add-on tr...
1,NCT00104715_Gravis_GETUG_EU'15,Author,Gemini 2.5 Flash (free-form),exact_match,Gravis G. et al.,Gwenaelle Gravis et al.,0.0,0.0,0.00,The names are similar but not an exact match.
2,NCT00104715_Gravis_GETUG_EU'15,Author,LandingAI (ADE),exact_match,Gravis G. et al.,Gwenaelle Gravis et al.,0.0,0.0,0.00,The authors are different.
3,NCT00104715_Gravis_GETUG_EU'15,COE_RCT_IND_OVERALL_RJ,Gemini 2.5 Flash (free-form),structured_text,Yes (randomized phase III),The overall risk judgment for the evidence pre...,0.0,0.0,0.00,The prediction provides a detailed explanation...
4,NCT00104715_Gravis_GETUG_EU'15,COE_RCT_IND_OVERALL_RJ,Gemini 2.5 Flash (native),structured_text,Yes (randomized phase III),not found,0.0,0.0,0.00,"The prediction ""not found"" is equivalent to an..."
...,...,...,...,...,...,...,...,...,...,...
921,NCT02799602_Smith_ARASENS_NEJM'22,Adverse Events - N (%) | All-Cause Grade 3 or ...,Gemini 2.5 Flash (native),numeric_tolerance,66.1%,431 (66.1%),0.5,1.0,0.75,GT contains one number: 66.1. Pred contains tw...
922,NCT02799602_Smith_ARASENS_NEJM'22,Adverse Events - N (%) | All-Cause Grade 3 or ...,LandingAI (ADE),numeric_tolerance,66.1%,431 (66.1%),1.0,0.5,0.75,The definition requires both count and percent...
923,NCT02799602_Smith_ARASENS_NEJM'22,Class of Agent in Treatment Arm 1,LandingAI (ADE),structured_text,Androgen-receptor inhibitor (darolutamide),Androgen-receptor inhibitor,1.0,0.5,0.75,The prediction correctly identifies the class ...
924,NCT02799602_Smith_ARASENS_NEJM'22,OS Rate (%) | Overall | Control,Gemini 2.5 Flash (native),numeric_tolerance,50.4% at 4 yr,"50.4% (95% CI, 46.3 to 54.6)",0.5,1.0,0.75,Since Pred has extra numbers that are not in G...
